In [ ]:
# ================================
#  POINT2TEXT
#  PointNet → DistilGPT2 decoder
# ===============================

In [ ]:
!pip install open3d --upgrade
!pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 80.3 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [ ]:
# ------------------------------------------------------------
# 0) SETTINGS
# ------------------------------------------------------------
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from tqdm import tqdm
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import os, sys, json, math, random
from pathlib import Path
import random
from transformers import DistilBertModel, DistilBertTokenizerFast

DRIVE_MODELNET_PATH = "/content/ModelNet40"   # IMPORTANT
EXTRACTED_DATASET_PATH = "/content/ModelNet40"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/ModelNet40.zip"
PRECOMPUTED_PATH = Path("/content/point2text_features.npz")
USE_NUM_POINTS = 1024
BATCH_SIZE = 128
EPOCHS = 3
EMBED_DIM = 256
LR = 1e-3
NUM_WORKERS = 2
SAVE_DIR = "/content/point2text_checkpoints"
RANDOM_SEED = 42

os.makedirs(SAVE_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


Device: cuda


In [ ]:
# --------- 1) Install deps (Colab) ----------
import sys, os, time, random
try:
    # quiet installs
    !pip install -q trimesh sentence-transformers matplotlib scikit-learn tqdm
except Exception as e:
    print("Install might have failed; continue if packages already present.", e)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 736.6/736.6 kB 26.7 MB/s eta 0:00:00


In [ ]:
# --------- 2a) Mount Google Drive (Colab) ----------
print("Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
# --------- 2b) Copy ZIP from Drive to Colab local disk ----------
import os, shutil

print("Copying ZIP file to Colab local disk (faster training)...")
local_zip = "/content/ModelNet40.zip"

if not os.path.exists(local_zip):
    shutil.copy(DRIVE_ZIP_PATH, local_zip)
else:
    print("ZIP already copied.")


Copying ZIP file to Colab local disk (faster training)...


In [ ]:
# --------- 2b) Unzip dataset ----------
print("Unzipping ModelNet40...")
if not os.path.exists(EXTRACTED_DATASET_PATH):
    os.makedirs(EXTRACTED_DATASET_PATH, exist_ok=True)
    !unzip -q "/content/ModelNet40.zip" -d "/content/"
else:
    print("Dataset already extracted.")

print("Dataset ready at:", EXTRACTED_DATASET_PATH)

Unzipping ModelNet40...
Dataset ready at: /content/ModelNet40


In [ ]:
# --------- 2c) Checkpoint dataset ----------
# Make checkpoint dir
os.makedirs(SAVE_DIR, exist_ok=True)
print("Checkpoint dir:", SAVE_DIR)

Checkpoint dir: /content/point2text_checkpoints


In [ ]:
# ============================================================
# 1) UTILITIES
# ============================================================
def load_off(file_path):
    """
    Load vertices from an OFF file.
    Handles:
    - Standard OFF header ("OFF")
    - Concatenated header like "OFF1234 567 890"
    - Missing OFF header
    """
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    # Handle header
    first_line = lines[0].upper()
    if first_line.startswith('OFF'):
        if first_line == 'OFF':
            # Standard OFF: next line has num_vertices
            header_line = lines[1]
            lines = lines[1:]
        else:
            # Concatenated header like "OFF18472 22860 45678"
            header_line = first_line[3:].strip()
            lines[0] = header_line
    else:
        # No OFF header
        header_line = first_line

    # Parse number of vertices
    parts = header_line.split()
    num_vertices = int(parts[0])

    # Read vertices
    vertices = np.array([list(map(float, lines[1 + j].split())) for j in range(num_vertices)], dtype=np.float32)
    return vertices

def normalize_pc(pc):
    pc = pc - np.mean(pc, axis=0, keepdims=True)
    m = np.max(np.sqrt(np.sum(pc**2, axis=1)))
    return pc / m

def batched_fps(points, k):
    B, N, _ = points.shape
    device = points.device
    sampled_idx = torch.zeros((B, k), dtype=torch.long, device=device)
    distances = torch.full((B, N), float('inf'), device=device)
    farthest = torch.randint(0, N, (B,), device=device)
    batch_idx = torch.arange(B, device=device)
    for i in range(k):
        sampled_idx[:, i] = farthest
        centroid = points[batch_idx, farthest].unsqueeze(1)
        dist = torch.sum((points - centroid) ** 2, dim=2)
        distances = torch.minimum(distances, dist)
        farthest = torch.argmax(distances, dim=1)
    return points[batch_idx.unsqueeze(1), sampled_idx]

# ============================================================
# 2) BUILD NPZ FAST
# ============================================================
def get_classes(modelnet_root):
    dirs = [d for d in os.listdir(modelnet_root) if os.path.isdir(os.path.join(modelnet_root, d))]
    return sorted([d for d in dirs if not d.startswith('.')])

def build_npz_fast(root_dir, out_file, classes, points_per_model=1024, val_ratio=0.1, device='cuda', batch_size=64):
    root_dir = Path(root_dir)
    all_points, all_labels = [], []
    file_list = []
    for cls_idx, cls in enumerate(classes):
        for split in ["train","test"]:
            off_dir = root_dir / cls / split
            if not off_dir.exists():
                continue
            for fname in sorted(os.listdir(off_dir)):
                if fname.endswith(".off"):
                    file_list.append((cls_idx, off_dir / fname))

    print(f"Total OFF files: {len(file_list)}")
    pbar = tqdm(total=len(file_list), desc="Building NPZ", unit="file")

    for i in range(0, len(file_list), batch_size):
        batch_files = file_list[i:i+batch_size]
        pcs, labels = [], []

        for cls_idx, fpath in batch_files:
            try:
                v = normalize_pc(load_off(fpath))
                if v.shape[0] >= points_per_model:
                    v_tensor = torch.tensor(v, dtype=torch.float32).unsqueeze(0).to(device)
                    sampled = batched_fps(v_tensor, points_per_model)[0].cpu().numpy()
                else:
                    idx = np.random.choice(v.shape[0], points_per_model, replace=True)
                    sampled = v[idx]
                pcs.append(torch.tensor(sampled, dtype=torch.float32))
                labels.append(cls_idx)
            except Exception as e:
                print("Failed:", fpath, e)
                continue

        if len(pcs) == 0:
            pbar.update(len(batch_files))
            continue

        pcs = torch.stack(pcs)
        all_points.append(pcs.numpy())
        all_labels.extend(labels)
        pbar.update(len(batch_files))

    pbar.close()
    if len(all_points) == 0:
        raise RuntimeError("No valid meshes found.")

    all_points = np.concatenate(all_points)
    all_labels = np.array(all_labels)

    idx = np.arange(len(all_points))
    np.random.shuffle(idx)
    train_n = int(len(all_points)*(1-val_ratio))

    np.savez(out_file,
             train_points=all_points[idx[:train_n]],
             train_labels=all_labels[idx[:train_n]],
             val_points=all_points[idx[train_n:]],
             val_labels=all_labels[idx[train_n:]])
    print(f"NPZ created: {out_file}, total models: {len(all_points)}")

# ============================================================
# 3) LOAD OR BUILD NPZ
# ============================================================
CLASS_NAMES = get_classes(DRIVE_MODELNET_PATH)
if not PRECOMPUTED_PATH.exists():
    print("No NPZ found — building with batched FPS...")
    build_npz_fast(DRIVE_MODELNET_PATH, PRECOMPUTED_PATH, CLASS_NAMES, points_per_model=USE_NUM_POINTS, device=device)
else:
    print("Using existing:", PRECOMPUTED_PATH)

data = np.load(PRECOMPUTED_PATH)
train_points = data["train_points"]
train_labels = data["train_labels"]
val_points   = data["val_points"]
val_labels   = data["val_labels"]
print("Train:", train_points.shape, "Val:", val_points.shape)

# ============================================================
# 4) DATASET & DATALOADER
# ============================================================
class ModelNetNPZ(Dataset):
    def __init__(self, pts, labels):
        self.pts = pts
        self.labels = labels
    def __len__(self):
        return len(self.pts)
    def __getitem__(self, i):
        pc = self.pts[i]
        idx = np.random.choice(pc.shape[0], USE_NUM_POINTS, replace=False)
        pc = pc[idx]
        return torch.tensor(pc, dtype=torch.float32), torch.tensor(self.labels[i], dtype=torch.long)

train_loader = DataLoader(ModelNetNPZ(train_points, train_labels),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(ModelNetNPZ(val_points, val_labels),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# ============================================================
# 5) MODEL DEFINITIONS
# ============================================================
class PointNetFeat(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, out_dim, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(out_dim)
    def forward(self, x):
        x = x.transpose(2,1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        x = torch.max(x, dim=2)[0]
        return x

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
text_model = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)

class TextEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.linear = nn.Linear(768, out_dim)
    def forward(self, texts):
        tok = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
        out = text_model(**tok).last_hidden_state[:,0]
        return self.linear(out)

class MLP(nn.Module):
    def __init__(self, d_in=256, d_out=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_in),
            nn.ReLU(),
            nn.Linear(d_in, d_out)
        )
    def forward(self, x):
        return self.net(x)

pc_encoder = PointNetFeat(EMBED_DIM).to(device)
txt_encoder = TextEncoder(EMBED_DIM).to(device)
pc_proj = MLP(EMBED_DIM, EMBED_DIM).to(device)
txt_proj = MLP(EMBED_DIM, EMBED_DIM).to(device)
params = list(pc_encoder.parameters()) + list(txt_encoder.parameters()) + list(pc_proj.parameters()) + list(txt_proj.parameters())
opt = torch.optim.Adam(params, lr=LR)
CLASS_DESC = [f"A 3D object from the class {c}." for c in CLASS_NAMES]

# ============================================================
# 6) PRECOMPUTE EMBEDDINGS
# ============================================================
if not PRECOMPUTED_PATH.exists():
    pc_encoder.eval(); txt_encoder.eval()
    train_pc_feats, val_pc_feats = [], []
    train_txt_feats, val_txt_feats = [], []
    with torch.no_grad():
        for pc, lbl in tqdm(train_loader, desc="Train PC->Feat"):
            pc = pc.to(device)
            pc_emb = pc_proj(pc_encoder(pc))
            train_pc_feats.append(pc_emb.cpu().numpy())
            text_batch = [CLASS_DESC[l.item()] for l in lbl]
            txt_emb = txt_proj(txt_encoder(text_batch))
            train_txt_feats.append(txt_emb.cpu().numpy())
        for pc, lbl in tqdm(val_loader, desc="Val PC->Feat"):
            pc = pc.to(device)
            pc_emb = pc_proj(pc_encoder(pc))
            val_pc_feats.append(pc_emb.cpu().numpy())
            text_batch = [CLASS_DESC[l.item()] for l in lbl]
            txt_emb = txt_proj(txt_encoder(text_batch))
            val_txt_feats.append(txt_emb.cpu().numpy())
    np.savez(PRECOMPUTED_PATH,
             train_pc=np.concatenate(train_pc_feats),
             train_txt=np.concatenate(train_txt_feats),
             val_pc=np.concatenate(val_pc_feats),
             val_txt=np.concatenate(val_txt_feats),
             train_labels=train_labels,
             val_labels=val_labels)
    print("Precomputed features saved:", PRECOMPUTED_PATH)
else:
# --- Load raw NPZ (from build_npz_fast / build_npz_batched) ---
    data = np.load(PRECOMPUTED_PATH)
    train_points = torch.tensor(data["train_points"], dtype=torch.float32)
    train_labels = torch.tensor(data["train_labels"], dtype=torch.long)
    val_points = torch.tensor(data["val_points"], dtype=torch.float32)
    val_labels = torch.tensor(data["val_labels"], dtype=torch.long)
    print("Raw points loaded:", train_points.shape, val_points.shape)

    # --- Precompute embeddings using your encoders ---
    pc_encoder.eval()
    txt_encoder.eval()
    train_pc_feats, val_pc_feats = [], []
    train_txt_feats, val_txt_feats = [], []

    CLASS_DESC = [f"A 3D object from the class {c}." for c in CLASS_NAMES]

    with torch.no_grad():
        # Train embeddings
        for i in range(0, len(train_points), BATCH_SIZE):
            pc_batch = train_points[i:i+BATCH_SIZE].to(device)
            pc_emb = pc_proj(pc_encoder(pc_batch))
            train_pc_feats.append(pc_emb.cpu())

            text_batch = [CLASS_DESC[l.item()] for l in train_labels[i:i+BATCH_SIZE]]
            txt_emb = txt_proj(txt_encoder(text_batch))
            train_txt_feats.append(txt_emb.cpu())

        # Validation embeddings
        for i in range(0, len(val_points), BATCH_SIZE):
            pc_batch = val_points[i:i+BATCH_SIZE].to(device)
            pc_emb = pc_proj(pc_encoder(pc_batch))
            val_pc_feats.append(pc_emb.cpu())

            text_batch = [CLASS_DESC[l.item()] for l in val_labels[i:i+BATCH_SIZE]]
            txt_emb = txt_proj(txt_encoder(text_batch))
            val_txt_feats.append(txt_emb.cpu())

    # Concatenate all embeddings
    train_pc = torch.cat(train_pc_feats, dim=0)
    train_txt = torch.cat(train_txt_feats, dim=0)
    val_pc = torch.cat(val_pc_feats, dim=0)
    val_txt = torch.cat(val_txt_feats, dim=0)

    print("Precomputed embeddings ready:", train_pc.shape, train_txt.shape)

# ============================================================
# 7) INFO-NCE LOSS & TRAINING LOOP
# ============================================================
def info_nce(pc_emb, txt_emb, temperature=0.07):
    """
    pc_emb: [B, D] tensor of point cloud embeddings
    txt_emb: [B, D] tensor of text embeddings
    """
    # Normalize embeddings
    pc_emb = F.normalize(pc_emb, dim=1)
    txt_emb = F.normalize(txt_emb, dim=1)

    # Cosine similarity
    logits = torch.matmul(pc_emb, txt_emb.T) / temperature
    labels = torch.arange(pc_emb.size(0), device=pc_emb.device)

    # Symmetric InfoNCE (both directions)
    loss_i2t = F.cross_entropy(logits, labels)        # pc -> txt
    loss_t2i = F.cross_entropy(logits.T, labels)      # txt -> pc

    return (loss_i2t + loss_t2i) / 2

SAVE_DIR = "/content/drive/MyDrive/point2text_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

start_epoch = 1
best_val_loss = float("inf")

# ----------------- Resume from latest checkpoint -----------------
ckpt_list = [f for f in os.listdir(SAVE_DIR)
             if f.endswith(".pt") and f.startswith("ckpt_") and f != "best_ckpt.pt"]

if ckpt_list:
    # Sort by epoch number (numeric), even if there are gaps
    def extract_epoch(fname):
        try:
            return int(fname.split('_')[1].split('.')[0])
        except:
            return -1  # ignore non-standard files

    latest_ckpt = sorted(ckpt_list, key=extract_epoch)[-1]
    ckpt_path = os.path.join(SAVE_DIR, latest_ckpt)
    print("Resuming from checkpoint:", ckpt_path)

    ckpt = torch.load(ckpt_path, map_location=device)
    pc_encoder.load_state_dict(ckpt["pc_encoder"])
    txt_encoder.load_state_dict(ckpt["txt_encoder"])
    pc_proj.load_state_dict(ckpt["pc_proj"])
    txt_proj.load_state_dict(ckpt["txt_proj"])
    opt.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    if "best_val_loss" in ckpt:
        best_val_loss = ckpt["best_val_loss"]

NUM_EPOCHS = 200
BATCH_SIZE = 128

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    # ----------------- Training -----------------
    pc_encoder.train(); txt_encoder.train()
    pc_proj.train(); txt_proj.train()
    total_loss = 0
    num_batches = 0

    for i in range(0, len(train_points), BATCH_SIZE):
        pc_batch = train_points[i:i+BATCH_SIZE].to(device)
        text_batch = [CLASS_DESC[l.item()] for l in train_labels[i:i+BATCH_SIZE]]

        pc_emb = pc_proj(pc_encoder(pc_batch))
        txt_emb = txt_proj(txt_encoder(text_batch))

        opt.zero_grad()
        loss = info_nce(pc_emb, txt_emb)
        loss.backward()
        opt.step()

        total_loss += loss.item()
        num_batches += 1

    avg_train_loss = total_loss / num_batches

    # ----------------- Validation -----------------
    pc_encoder.eval(); txt_encoder.eval()
    pc_proj.eval(); txt_proj.eval()
    val_loss_total = 0
    val_batches = 0

    with torch.no_grad():
        for i in range(0, len(val_points), BATCH_SIZE):
            pc_batch = val_points[i:i+BATCH_SIZE].to(device)
            text_batch = [CLASS_DESC[l.item()] for l in val_labels[i:i+BATCH_SIZE]]

            pc_emb = pc_proj(pc_encoder(pc_batch))
            txt_emb = txt_proj(txt_encoder(text_batch))

            val_loss_total += info_nce(pc_emb, txt_emb).item()
            val_batches += 1

    avg_val_loss = val_loss_total / val_batches
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # ----------------- Save checkpoints -----------------
    ckpt_path = os.path.join(SAVE_DIR, f"ckpt_{epoch}.pt")
    torch.save({
        "epoch": epoch,
        "pc_encoder": pc_encoder.state_dict(),
        "txt_encoder": txt_encoder.state_dict(),
        "pc_proj": pc_proj.state_dict(),
        "txt_proj": txt_proj.state_dict(),
        "optimizer": opt.state_dict(),
        "best_val_loss": best_val_loss
    }, ckpt_path)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_path = os.path.join(SAVE_DIR, "best_ckpt.pt")
        torch.save({
            "epoch": epoch,
            "pc_encoder": pc_encoder.state_dict(),
            "txt_encoder": txt_encoder.state_dict(),
            "pc_proj": pc_proj.state_dict(),
            "txt_proj": txt_proj.state_dict(),
            "optimizer": opt.state_dict(),
            "best_val_loss": best_val_loss
        }, best_path)
        print(f"Best checkpoint updated: {best_path}")

No NPZ found — building with batched FPS...
Total OFF files: 12311


Building NPZ: 100%|██████████| 12311/12311 [28:31<00:00,  7.19file/s]


NPZ created: /content/point2text_features.npz, total models: 12311
Train: (11079, 1024, 3) Val: (1232, 1024, 3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Raw points loaded: torch.Size([11079, 1024, 3]) torch.Size([1232, 1024, 3])
Precomputed embeddings ready: torch.Size([11079, 256]) torch.Size([11079, 256])
Resuming from checkpoint: /content/drive/MyDrive/point2text_checkpoints/ckpt_200.pt


In [ ]:
# TOP-1 AND TOP-5 RETRIEVAL ACCURACY
# Move embeddings to device
train_pc = train_pc.to(device)
train_txt = train_txt.to(device)
val_pc   = val_pc.to(device)
val_txt  = val_txt.to(device)
train_labels = train_labels.to(device)
val_labels   = val_labels.to(device)

def retrieval_accuracy(pc_emb, txt_emb, labels, topk=(1,5)):
    """
    Compute top-k retrieval accuracy.
    For each point cloud, we find the closest text embedding by cosine similarity.
    """
    # Normalize embeddings
    pc_norm = F.normalize(pc_emb, dim=1)
    txt_norm = F.normalize(txt_emb, dim=1)

    # Cosine similarity: [N_pc, N_txt]
    sim_matrix = pc_norm @ txt_norm.T

    topk_values, topk_indices = sim_matrix.topk(max(topk), dim=1)
    correct = labels.unsqueeze(1) == train_labels[topk_indices]

    acc = {}
    for k in topk:
        acc[k] = correct[:, :k].any(dim=1).float().mean().item() * 100
    return acc

# Evaluate on validation set
topk_accs = retrieval_accuracy(val_pc, train_txt, val_labels, topk=(1,5))
print(f"Top-1 Accuracy: {topk_accs[1]:.2f}%")
print(f"Top-5 Accuracy: {topk_accs[5]:.2f}%")

Top-1 Accuracy: 2.35%
Top-5 Accuracy: 2.35%


In [ ]:
# ============================================================
#  INFERENCE DEMO — Point Cloud → Natural Language Description
# ============================================================

from torch.nn.functional import cosine_similarity

pc_encoder.eval(); pc_proj.eval()
txt_encoder.eval(); txt_proj.eval()

def describe_pointcloud(pc_np):
    """
    pc_np: numpy array (N,3)
    """

    # Random sampling (same as training)
    idx = np.random.choice(pc_np.shape[0], USE_NUM_POINTS, replace=False)
    pc_np = pc_np[idx]

    pc = torch.tensor(pc_np, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        pc_emb = pc_proj(pc_encoder(pc))  # (1, D)
        pc_emb = F.normalize(pc_emb, dim=1)

    # Generate 40 candidate text embeddings
    texts = [f"A 3D object from the class {c}." for c in CLASS_NAMES]

    with torch.no_grad():
        txt_emb = txt_proj(txt_encoder(texts))  # (40, D)
        txt_emb = F.normalize(txt_emb, dim=1)

    sims = cosine_similarity(pc_emb, txt_emb).cpu().numpy().flatten()

    top_idx = sims.argmax()
    return CLASS_NAMES[top_idx], sims[top_idx]

# -----------------------------
# Test on one random val sample
# -----------------------------
sample_i = np.random.randint(len(val_points))
pc_sample = val_points[sample_i]

pred_class, score = describe_pointcloud(pc_sample)
true_class = CLASS_NAMES[val_labels[sample_i]]

print("Inference demo")
print("-------------------------")
print("True class :", true_class)
print("Predicted  :", pred_class)
print("Similarity :", float(score))


Inference demo
-------------------------
True class : bookshelf
Predicted  : bookshelf
Similarity : 0.6582647562026978


/tmp/ipython-input-628102268.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pc = torch.tensor(pc_np, dtype=torch.float32).unsqueeze(0).to(device)


In [ ]:
# ============================================================
#  INFERENCE DEMO — Point Cloud → Natural Language Description
#  Interactive 3D visualization in Colab using Plotly
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np
import plotly.graph_objects as go

# -----------------------------
# Set models to eval mode
# -----------------------------
pc_encoder.eval()
pc_proj.eval()
txt_encoder.eval()
txt_proj.eval()

# -----------------------------
# Function to describe a point cloud
# -----------------------------
def describe_pointcloud(pc_np):
    """
    pc_np: numpy array (N,3)
    """
    # Random sampling (same as training)
    idx = np.random.choice(pc_np.shape[0], USE_NUM_POINTS, replace=False)
    pc_np = pc_np[idx]

    pc = torch.tensor(pc_np, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        pc_emb = pc_proj(pc_encoder(pc))  # (1, D)
        pc_emb = F.normalize(pc_emb, dim=1)

    # Generate candidate text embeddings
    texts = [f"A 3D object from the class {c}." for c in CLASS_NAMES]
    with torch.no_grad():
        txt_emb = txt_proj(txt_encoder(texts))  # (num_classes, D)
        txt_emb = F.normalize(txt_emb, dim=1)

    sims = F.cosine_similarity(pc_emb, txt_emb).cpu().numpy().flatten()
    top_idx = sims.argmax()
    return CLASS_NAMES[top_idx], float(sims[top_idx])

# -----------------------------
# Function to plot point cloud interactively
# -----------------------------
def plot_pointcloud(pc_np, title="Point Cloud"):
    x, y, z = pc_np[:,0], pc_np[:,1], pc_np[:,2]
    fig = go.Figure(data=[go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=3, color=z, colorscale='Viridis')
    )])
    fig.update_layout(title=title, width=700, height=600)
    fig.show()

# -----------------------------
# Demo on a random validation sample
# -----------------------------
sample_i = np.random.randint(len(val_points))
pc_sample = val_points[sample_i]

pred_class, score = describe_pointcloud(pc_sample)
true_class = CLASS_NAMES[val_labels[sample_i]]

print("Inference Demo")
print("-------------------------")
print("True class :", true_class)
print("Predicted  :", pred_class)
print("Similarity :", score)

# Plot the 3D point cloud
plot_pointcloud(pc_sample, title=f"True: {true_class} | Predicted: {pred_class}")

Inference Demo
-------------------------
True class : toilet
Predicted  : toilet
Similarity : 0.7162226438522339


/tmp/ipython-input-2855257469.py:30: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [ ]:
# ============================================================
# SIDE-BY-SIDE INFERENCE DEMO — Point Cloud → Natural Language Description
# Interactive 3D visualization using Plotly
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, HTML

# -----------------------------
# Assumes your models are already loaded:
# pc_encoder, pc_proj, txt_encoder, txt_proj
# val_points, val_labels, CLASS_NAMES, USE_NUM_POINTS, device
# -----------------------------

pc_encoder.eval()
pc_proj.eval()
txt_encoder.eval()
txt_proj.eval()

def describe_pointcloud(pc_tensor, top_k=3):
    """
    pc_tensor: PyTorch tensor of shape (N,3) or NumPy array
    """
    # Convert to NumPy if needed
    if isinstance(pc_tensor, torch.Tensor):
        pc_np = pc_tensor.detach().cpu().numpy()
    else:
        pc_np = pc_tensor

    # Random sampling
    idx = np.random.choice(pc_np.shape[0], USE_NUM_POINTS, replace=False)
    pc_sampled = pc_np[idx]

    pc_tensor_sampled = torch.tensor(pc_sampled, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        pc_emb = pc_proj(pc_encoder(pc_tensor_sampled))  # (1, D)
        pc_emb = F.normalize(pc_emb, dim=1)

    # Candidate text embeddings
    texts = [f"A 3D object from the class {c}." for c in CLASS_NAMES]
    with torch.no_grad():
        txt_emb = txt_proj(txt_encoder(texts))  # (num_classes, D)
        txt_emb = F.normalize(txt_emb, dim=1)

    sims = F.cosine_similarity(pc_emb, txt_emb).cpu().numpy().flatten()
    top_idx = sims.argsort()[-top_k:][::-1]  # top-k indices
    top_classes = [CLASS_NAMES[i] for i in top_idx]
    top_scores = sims[top_idx]
    top_descriptions = [f"{c} (sim={s:.3f})" for c,s in zip(top_classes, top_scores)]

    return top_classes, top_scores, top_descriptions, pc_sampled

def plot_side_by_side(gt_pc, pred_pc, gt_class, top_classes, top_scores, sample_points=1024):
    """
    Plots ground truth and predicted point clouds side by side with captions.
    """
    # Sample points if needed
    if gt_pc.shape[0] > sample_points:
        idx = np.random.choice(gt_pc.shape[0], sample_points, replace=False)
        gt_pc = gt_pc[idx]

    if pred_pc.shape[0] > sample_points:
        idx = np.random.choice(pred_pc.shape[0], sample_points, replace=False)
        pred_pc = pred_pc[idx]

    # Create scatter3d for GT
    gt_scatter = go.Scatter3d(
        x=gt_pc[:,0], y=gt_pc[:,1], z=gt_pc[:,2],
        mode='markers',
        marker=dict(size=2, color=gt_pc[:,2], colorscale='Blues'),
        name='Ground Truth'
    )

    # Caption
    gt_caption = f"<b>Ground Truth:</b> {gt_class}"

    # Create scatter3d for prediction
    pred_scatter = go.Scatter3d(
        x=pred_pc[:,0], y=pred_pc[:,1], z=pred_pc[:,2],
        mode='markers',
        marker=dict(size=2, color=pred_pc[:,2], colorscale='Reds'),
        name='Prediction'
    )

    # Caption
    pred_caption = "<b>Prediction:</b><br>" + "<br>".join([f"{c}: {s:.3f}" for c,s in zip(top_classes, top_scores)])

    # Combine in a single figure with subplots
    fig = go.Figure(data=[gt_scatter, pred_scatter])
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z')
        ),
        width=1200,
        height=600,
        title_text=f"{gt_caption} | {pred_caption}",
        title_x=0.5
    )
    fig.show()

# -----------------------------
# Run demo on one random validation sample
# -----------------------------
sample_i = np.random.randint(len(val_points))
gt_pc = val_points[sample_i]
gt_class = CLASS_NAMES[val_labels[sample_i]]

top_classes, top_scores, top_descriptions, pred_pc = describe_pointcloud(gt_pc, top_k=3)

print(f"True class: {gt_class}")
print(f"Top prediction: {top_classes[0]} (sim={top_scores[0]:.3f})")

plot_side_by_side(gt_pc, pred_pc, gt_class, top_classes, top_scores, sample_points=USE_NUM_POINTS)




True class: stool
Top prediction: stool (sim=0.724)
